<a href="https://colab.research.google.com/github/swarnendu-sekhar-das/AIT511-Project-I/blob/main/NLP_Telecom_QA_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nest_asyncio
import uvicorn
import threading
import traceback
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata

conn = sqlite3.connect(SQLITE_PATH, check_same_thread=False)
cursor = conn.cursor()

# 1. API Schema
class QueryRequest(BaseModel):
    query: str

class SopResponse(BaseModel):
    sop_id: str
    vendor: str
    severity: str
    title: str

class QueryResponse(BaseModel):
    answer: str
    retrieved_sops: list[SopResponse]

# 2. Initialize FastAPI
app = FastAPI(title="SecureNOC RAG API", version="1.0")

# 3. POST Endpoint
@app.post("/ask", response_model=QueryResponse)
async def ask_securenoc(request: QueryRequest):
    try:
        print(f"\nReceived query from frontend: {request.query}")

        # Call existing pipeline
        ans, srcs = process_query(request.query)

        print("Query processed successfully! Sending back to frontend...")

        # Format the sources cleanly for the JSON response
        formatted_srcs = []
        for s in srcs:
            formatted_srcs.append(SopResponse(
                sop_id=s.get("sop_id", "N/A"),
                vendor=s.get("vendor", "Unknown"),
                severity=s.get("severity", "UNKNOWN"),
                title=s.get("title", "Untitled")
            ))

        return QueryResponse(answer=ans, retrieved_sops=formatted_srcs)

    except Exception as e:
        print("\nFATAL ERROR IN PROCESS_QUERY:")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

# 4. Expose the Server to the Internet
ngrok_token = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(ngrok_token)

# Close any existing tunnels to prevent errors
ngrok.kill()

public_url = ngrok.connect(8000).public_url
print("="*60)
print(f"SECURENOC API IS LIVE AT: {public_url}/ask")
print("="*60)

# 5. Start the Server in a Background Thread
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print("Background thread started. API is ready to receive requests!")

SecretNotFoundError: Secret NGROK_AUTHTOKEN does not exist.

In [ ]:
import nest_asyncio
import uvicorn
import threading
import traceback
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata

conn = sqlite3.connect(SQLITE_PATH, check_same_thread=False)
cursor = conn.cursor()

# 1. API Schema
class QueryRequest(BaseModel):
    query: str

class SopResponse(BaseModel):
    sop_id: str
    vendor: str
    severity: str
    title: str

class QueryResponse(BaseModel):
    answer: str
    retrieved_sops: list[SopResponse]

# 2. Initialize FastAPI
app = FastAPI(title="SecureNOC RAG API", version="1.0")

# 3. POST Endpoint
@app.post("/ask", response_model=QueryResponse)
async def ask_securenoc(request: QueryRequest):
    try:
        print(f"\nReceived query from frontend: {request.query}")

        # Call existing pipeline
        ans, srcs = process_query(request.query)

        print("Query processed successfully! Sending back to frontend...")

        # Format the sources cleanly for the JSON response
        formatted_srcs = []
        for s in srcs:
            formatted_srcs.append(SopResponse(
                sop_id=s.get("sop_id", "N/A"),
                vendor=s.get("vendor", "Unknown"),
                severity=s.get("severity", "UNKNOWN"),
                title=s.get("title", "Untitled")
            ))

        return QueryResponse(answer=ans, retrieved_sops=formatted_srcs)

    except Exception as e:
        print("\nFATAL ERROR IN PROCESS_QUERY:")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

# 4. Expose the Server to the Internet
ngrok_token = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(ngrok_token)

# Close any existing tunnels to prevent errors
ngrok.kill()

public_url = ngrok.connect(8000).public_url
print(f"SECURENOC API IS LIVE AT: {public_url}/ask")

# 5. Start the Server in a Background Thread
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print("Background thread started. API is ready to receive requests!")

SECURENOC API IS LIVE AT: https://mia-propertied-cristopher.ngrok-free.dev/ask
Background thread started. API is ready to receive requests!


INFO:     Started server process [23216]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
import os
import json
import random
import time
from tqdm import tqdm
from google import genai
from google.genai import types
from google.colab import userdata

# 1. API CONFIGURATION
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = input("Enter your Gemini API Key: ")

# Initialize the new SDK client
client = genai.Client(api_key=API_KEY)

# 2. THE "TRAP" PERMUTATION MATRIX
VENDORS = ["Cisco", "Juniper", "Nokia", "Huawei", "Ericsson", "ZTE"]

REGIONS = [
    "National Core",
    "IP/MPLS Backbone",
    "EPC Core",
    "Metro Aggregation",
    "International Gateway"
]

TRAP_CONCEPTS = [
    {"term": "S-GW (Serving Gateway)", "confused_with": "SGW (Security Gateway)", "severity": "MAJOR", "impact": "Drops active LTE handovers across regions."},
    {"term": "P-GW (Packet Gateway)", "confused_with": "S-GW", "severity": "CRITICAL", "impact": "Nationwide mobile data outage."},
    {"term": "MME (Mobility Management Entity)", "confused_with": "SGSN", "severity": "CRITICAL", "impact": "UE attach failures in LTE network."},
    {"term": "SGSN", "confused_with": "MME", "severity": "MAJOR", "impact": "3G session handling disruption."},
    {"term": "GTP-C", "confused_with": "GTP-U", "severity": "CRITICAL", "impact": "Session signaling OK but no user data flow."},
    {"term": "GTP-U", "confused_with": "GTP-C", "severity": "CRITICAL", "impact": "User traffic dropped despite active sessions."},
    {"term": "LDP", "confused_with": "RSVP-TE", "severity": "MAJOR", "impact": "MPLS label distribution inconsistency."},
    {"term": "RSVP-TE", "confused_with": "LDP", "severity": "MAJOR", "impact": "Traffic engineering tunnels not established."},
    {"term": "OSPF Area 0", "confused_with": "Non-Backbone Area", "severity": "CRITICAL", "impact": "Routing domain partition in backbone."},
    {"term": "BGP Route Reflector", "confused_with": "BGP Confederation", "severity": "MAJOR", "impact": "iBGP route propagation issues."}
]

unique_scenarios = []
print("Generating the Stress Test Matrix:")

while len(unique_scenarios) < 750:
    vendor = random.choice(VENDORS)
    region = random.choice(REGIONS)
    trap = random.choice(TRAP_CONCEPTS)
    error_code = f"ERR-{random.randint(1000, 9999)}"

    combo = f"{vendor}-{region}-{trap['term']}-{error_code}"
    if combo not in [s['id_string'] for s in unique_scenarios]:
        unique_scenarios.append({
            "id_string": combo,
            "vendor": vendor,
            "region": region,
            "term": trap['term'],
            "confused_with": trap['confused_with'],
            "severity": trap['severity'],
            "impact": trap['impact'],
            "error_code": error_code
        })

# 3. THE "BLAST RADIUS" PROMPT & GENERATION FUNCTION
PROMPT_TEMPLATE = """
You are a Senior NOC Engineer writing an SOP for a Tier-1 Telecom.
Write a highly technical SOP for the following exact scenario.

**CRITICAL INSTRUCTION:** This SOP is for {term}. It is vital that this is NOT confused with {confused_with}.
The CLI commands and safety warnings must accurately reflect the specific impact of {term}.

**TARGET SYSTEM:** {term} deployed on {vendor} equipment in the {region} region.
**REAL-WORLD IMPACT:** {impact}
**SEVERITY:** {severity}
**ERROR CODE:** {error_code}

Output EXACTLY this JSON structure:
{{
  "sop_id": "SOP-TRAP-{error_code}",
  "title": "SOP for {term} Disruption",
  "vendor": "{vendor}",
  "severity": "{severity}",
  "estimated_time": "[Time in minutes]",
  "prerequisites": ["List 2-3 specific technical prerequisites"],
  "safety_warnings": [
    "Write a strong warning explicitly stating the blast radius: {impact}",
    "Include a warning explicitly telling the engineer NOT to confuse this with {confused_with}"
  ],
  "steps": [
    {{
      "step_number": 1,
      "phase": "Diagnosis",
      "action": "Technical description of the action",
      "command": "Exact CLI command (Must be accurate for {vendor} equipment resolving {term})",
      "expected_output": "What the engineer should look for"
    }}
  ]
}}
Generate exactly 4 to 6 steps. Ensure commands match the vendor's syntax.
"""

def generate_trap_sop(scenario, client, max_retries=3):
    prompt = PROMPT_TEMPLATE.format(**scenario)

    for attempt in range(max_retries):
        try:
            # Using Gemma 3 27B IT, but REMOVED the strict JSON mime-type config
            response = client.models.generate_content(
                model='gemma-3-27b-it',
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.2 # Keep this low for strict formatting
                )
            )

            # 1. Get the raw text
            raw_text = response.text.strip()

            # 2. Strip out Markdown code blocks if the LLM adds them
            if raw_text.startswith("```json"):
                raw_text = raw_text[7:]
            if raw_text.startswith("```"):
                raw_text = raw_text[3:]
            if raw_text.endswith("```"):
                raw_text = raw_text[:-3]

            raw_text = raw_text.strip()

            # 3. Parse it
            return json.loads(raw_text)

        except json.JSONDecodeError:
            print(f"\nJSON Parsing Error on {scenario['term']}. The LLM didn't format it perfectly. Retrying... (Attempt {attempt + 1}/{max_retries})")
            time.sleep(3)
        except Exception as e:
            error_msg = str(e)
            if '429' in error_msg or 'RESOURCE_EXHAUSTED' in error_msg:
                print(f"\nRate limit hit on {scenario['term']}. Sleeping for 30s... (Attempt {attempt + 1}/{max_retries})")
                time.sleep(30)
            else:
                print(f"\nUnrecoverable API Error on {scenario['term']}: {e}")
                return None

    print(f"\nFailed to generate SOP for {scenario['term']} after {max_retries} attempts.")
    return None

# 4. EXECUTION PIPELINE WITH INCREMENTAL SAVING
dataset = []
output_file = "telecom_sops_500_stress_test.json"
print("Starting Fault Tolerant Telecom Trap SOP Generation...")

# Load existing progress if the runtime crashed previously
if os.path.exists(output_file):
    with open(output_file, "r") as f:
        dataset = json.load(f)
    print(f"Resuming from existing dataset with {len(dataset)} SOPs.")

    # Filter out scenarios that have already been generated
    generated_ids = [sop.get('sop_id') for sop in dataset if 'sop_id' in sop]
    unique_scenarios = [s for s in unique_scenarios if f"SOP-TRAP-{s['error_code']}" not in generated_ids]

for i, scenario in tqdm(enumerate(unique_scenarios), total=len(unique_scenarios), desc="Generating Traps"):
    sop_data = generate_trap_sop(scenario, client)

    if sop_data:
        steps_text = " ".join([f"Step {s.get('step_number', '')}: {s.get('action', '')} Command: {s.get('command', '')}" for s in sop_data.get('steps', [])])
        sop_data["search_content"] = f"Vendor: {sop_data.get('vendor')} | Severity: {sop_data.get('severity')} | Target: {scenario['term']} | Steps: {steps_text}"

        dataset.append(sop_data)

        # INCREMENTAL SAVE: Write to disk after every successful generation
        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

    # Sleep to respect the Gemma 3 27B 30 RPM limit (1 request every 2 seconds, using 3 to be safe)
    time.sleep(3)

print(f"\nSuccessfully saved {len(dataset)} Stress Test SOPs to {output_file}!")

Generating the Stress Test Matrix:
Starting Fault Tolerant Telecom Trap SOP Generation...
Resuming from existing dataset with 100 SOPs.


Generating Traps:  84%|████████▍ | 81/96 [30:56<06:10, 24.71s/it]


JSON Parsing Error on GTP-U. The LLM didn't format it perfectly. Retrying... (Attempt 1/3)


Generating Traps: 100%|██████████| 96/96 [37:20<00:00, 23.34s/it]


Successfully saved 196 Stress Test SOPs to telecom_sops_500_stress_test.json!


In [ ]:
# PHASE 1: TELECOM NOC PROCEDURAL RAG
!pip install -q datasets sentence-transformers chromadb pandas rank_bm25 transformers accelerate bitsandbytes

import torch
import chromadb
import pandas as pd
import string
import textwrap
import gc
import numpy as np
import os
import shutil
import json
import sqlite3
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Accelerator: {device}")

Hardware Accelerator: cuda


In [ ]:
# DB_PATH = "./telecom_vector_db"
# SQLITE_PATH = "telecom_sops.db"

import uuid

DB_PATH = f"/tmp/telecom_vector_db_{uuid.uuid4().hex}"
SQLITE_PATH = f"/tmp/telecom_sops_{uuid.uuid4().hex}.db"

# 1. Clean up old runs
if os.path.exists(DB_PATH): shutil.rmtree(DB_PATH)
if os.path.exists(SQLITE_PATH): os.remove(SQLITE_PATH)

# 2. Initialize SQLite (for Document Store)
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sops (
        sop_id TEXT PRIMARY KEY,
        vendor TEXT,
        severity TEXT,
        full_json TEXT
    )
''')
conn.commit()
print("SQLite initialized.")

# 3. Initialize ChromaDB (for Vector Store)
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection(
    name="telecom_sops",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB initialized.")

# 4. Load Embedding Model
print("🔹 Loading SOTA Embedding Model: BAAI/bge-base-en-v1.5")
embedding_func = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

SQLite initialized.
ChromaDB initialized.
🔹 Loading SOTA Embedding Model: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print("Loading structured SOPs and rewriting for dense embeddings:")

with open("telecom_sops_500_stress_test.json", "r") as f:
    dataset = json.load(f)

bm25_tokenized_corpus = []
batch_docs = []
batch_metadatas = []
batch_ids = []

seen_ids = set()

def simple_tokenize(text):
    return text.lower().translate(str.maketrans('', '', string.punctuation)).split()

for sop in tqdm(dataset, desc="Indexing"):
    sop_id = sop["sop_id"]

    if sop_id in seen_ids:
        continue
    seen_ids.add(sop_id)

    vendor = sop.get("vendor", "Unknown")
    severity = sop.get("severity", "UNKNOWN")

    # Conversational Data Representation
    term = sop.get('title', '').replace('SOP for ', '').replace(' Disruption', '')
    prose_content = f"This is a {severity} severity Standard Operating Procedure (SOP) for {vendor} equipment. "
    prose_content += f"It resolves disruptions caused by {term}. The procedure involves the following steps: "

    steps_text = " ".join([f"Step {s.get('step_number')}: {s.get('action')}. Execute command: `{s.get('command')}`." for s in sop.get('steps', [])])
    prose_content += steps_text

    # Override the original rigid search_content
    sop["search_content"] = prose_content

    # 1. Insert into SQLite (Full Document)
    cursor.execute(
        "INSERT INTO sops (sop_id, vendor, severity, full_json) VALUES (?, ?, ?, ?)",
        (sop_id, vendor, severity, json.dumps(sop))
    )

    # 2. Prepare for Vector DB & BM25
    bm25_tokenized_corpus.append(simple_tokenize(prose_content))
    batch_docs.append(prose_content)
    batch_ids.append(sop_id)
    batch_metadatas.append({"vendor": vendor, "severity": severity})

conn.commit()

# 3. Insert into ChromaDB
embeddings = embedding_func.encode(batch_docs, normalize_embeddings=True).tolist()
collection.add(
    documents=batch_docs,
    embeddings=embeddings,
    metadatas=batch_metadatas,
    ids=batch_ids
)

print(f"Indexed {collection.count()} SOPs with optimized prose embeddings.")

print("Building BM25 Index...")
bm25 = BM25Okapi(bm25_tokenized_corpus)
del bm25_tokenized_corpus
gc.collect()

Loading structured SOPs and rewriting for dense embeddings:


Indexing: 100%|██████████| 196/196 [00:00<00:00, 3317.33it/s]


Indexed 196 SOPs with optimized prose embeddings.
Building BM25 Index...


3786

In [ ]:
print("Loading SOTA Cross-Encoder: BAAI/bge-reranker-base")
reranker = CrossEncoder('BAAI/bge-reranker-base', device=device)

print("\nLoading Llama-3-8B (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    temperature=0.0
)

print("AI Models Loaded Successfully. Greedy Decoding Enforced.")

Loading SOTA Cross-Encoder: BAAI/bge-reranker-base


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


AI Models Loaded Successfully. Greedy Decoding Enforced.


In [ ]:
!pip install -q gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 7.2 MB/s eta 0:00:00


In [ ]:
from gliner import GLiNER
import numpy as np
import json

# 1. LOAD GLiNER (Zero-Shot Metadata Extractor)
print("🔍 Loading GLiNER for localized, zero-shot metadata extraction...")
# Load the model weights
ner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")

ner_model = ner_model.to(device)
def extract_metadata_gliner(query):
    labels = ["telecom vendor", "severity level"]

    # Predict entities directly from the raw string
    entities = ner_model.predict_entities(query, labels)

    filters = {}
    for ent in entities:
        text_val = ent["text"].lower()
        label = ent["label"]

        # Clean and normalize the extracted text to match ChromaDB metadata
        if label == "telecom vendor":
            for v in ["cisco", "juniper", "nokia", "ericsson", "huawei", "palo alto", "fortinet", "ciena"]:
                if v in text_val:
                    filters["vendor"] = v.capitalize()
                    break
        elif label == "severity level":
            for s in ["critical", "major", "minor", "warning"]:
                if s in text_val:
                    filters["severity"] = s.upper()
                    break

    return filters if filters else None

# THE OPTIMIZED QUERY PIPELINE
def process_query(query):
    # Extract Metadata Filters using Local GLiNER
    chroma_filters = extract_metadata_gliner(query)
    print(f"  [Diagnostics] Router applied filters: {chroma_filters}")

    # Dense Retrieval
    query_emb = embedding_func.encode([f"Represent this sentence for searching relevant passages: {query}"], normalize_embeddings=True).tolist()

    dense_res = collection.query(
        query_embeddings=query_emb,
        n_results=10,
        where=chroma_filters
    )

    if not dense_res['ids'][0]:
        return "No matching SOPs found for this specific hardware/fault.", []

    dense_hits = {id: score for id, score in zip(dense_res['ids'][0], dense_res['distances'][0])}

    # Sparse Retrieval (BM25)
    tokenized_query = simple_tokenize(query)
    sparse_scores = bm25.get_scores(tokenized_query)
    top_sparse_indices = np.argsort(sparse_scores)[-10:][::-1]

    # Tuned Reciprocal Rank Fusion (RRF) for dense datasets
    fusion_scores = {}
    k = 20 # Aggressively favor the top hits to reduce noise before the Cross-Encoder

    for rank, (doc_id, _) in enumerate(dense_hits.items()):
        fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

    for rank, idx in enumerate(top_sparse_indices):
        doc_id = batch_ids[idx]
        fusion_scores[doc_id] = fusion_scores.get(doc_id, 0) + 1 / (k + rank + 1)

    sorted_candidates = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    top_ids = [doc_id for doc_id, _ in sorted_candidates]

    # Fetch FULL JSON from decoupled SQLite storage
    placeholders = ','.join(['?'] * len(top_ids))
    cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
    rows = cursor.fetchall()

    id_to_json = {row[0]: json.loads(row[1]) for row in rows}
    candidate_docs = [id_to_json[uid] for uid in top_ids if uid in id_to_json]

    # Reranking via Cross-Encoder
    pairs = [[query, doc["search_content"]] for doc in candidate_docs]
    scores = reranker.predict(pairs)
    ranked_final = sorted(list(zip(candidate_docs, scores)), key=lambda x: x[1], reverse=True)

    # Expanded Context Window (Taking Top 3 instead of Top 2)
    final_sops = [doc for doc, score in ranked_final][:3]

    # Format Context for Operational Safety
    context_blocks = []
    for sop in final_sops:
        block = f"SOP ID: {sop['sop_id']} | Vendor: {sop.get('vendor')} | Severity: {sop.get('severity')}\n"
        block += f"Warnings: {', '.join(sop.get('safety_warnings', []))}\nSteps:\n"
        for step in sop.get('steps', []):
            block += f"  {step['step_number']}. {step['action']} -> Command: `{step['command']}`\n"
        context_blocks.append(block)

    context_string = "\n\n".join(context_blocks)

    # Strict Citation Prompting to prevent Hallucinations
    # Strict XML Chain-of-Thought Prompting
    # Robust Chain-of-Thought Prompting for 4-bit Models
    sys_prompt = (
        "You are a factual Tier-1 NOC AI Assistant. You receive multiple SOPs. "
        "You must follow these instructions exactly:\n"
        "1. Identify the SINGLE most relevant SOP for the user's issue.\n"
        "2. First, think step-by-step about why this SOP is correct. Prefix this section with 'REASONING:'.\n"
        "3. Second, provide your final procedural answer. Prefix this section with 'FINAL_ANSWER:'.\n"
        "4. In your final answer, do NOT generate artificial warnings unless the chosen SOP explicitly states them.\n"
        "5. In your final answer, append the exact SOP ID in brackets for EVERY CLI command (e.g., `clear ip bgp *` [SOP-123])."
    )

    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Context SOPs:\n{context_string}\n\nUser Issue: {query}"},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # We remove max_length to avoid the Transformers conflict warning
    outputs = llm_pipe(prompt)

    full_out = outputs[0]["generated_text"]
    generated_text = full_out.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

    # Fail proof extraction: Split the string and take everything after the marker
    if "FINAL_ANSWER:" in generated_text:
        answer = generated_text.split("FINAL_ANSWER:")[-1].strip()
    else:
        # Emergency fallback if the model completely ignores formatting
        answer = generated_text

    return answer, final_sops

🔍 Loading GLiNER for localized, zero-shot metadata extraction...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic

In [ ]:
import nest_asyncio
import uvicorn
import threading
import traceback
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata

conn = sqlite3.connect(SQLITE_PATH, check_same_thread=False)
cursor = conn.cursor()

# 1. API Schema
class QueryRequest(BaseModel):
    query: str

class SopResponse(BaseModel):
    sop_id: str
    vendor: str
    severity: str
    title: str

class QueryResponse(BaseModel):
    answer: str
    retrieved_sops: list[SopResponse]

# 2. Initialize FastAPI
app = FastAPI(title="SecureNOC RAG API", version="1.0")

# 3. POST Endpoint
@app.post("/ask", response_model=QueryResponse)
async def ask_securenoc(request: QueryRequest):
    try:
        print(f"\nReceived query from frontend: {request.query}")

        # Call existing pipeline
        ans, srcs = process_query(request.query)

        print("Query processed successfully! Sending back to frontend...")

        # Format the sources cleanly for the JSON response
        formatted_srcs = []
        for s in srcs:
            formatted_srcs.append(SopResponse(
                sop_id=s.get("sop_id", "N/A"),
                vendor=s.get("vendor", "Unknown"),
                severity=s.get("severity", "UNKNOWN"),
                title=s.get("title", "Untitled")
            ))

        return QueryResponse(answer=ans, retrieved_sops=formatted_srcs)

    except Exception as e:
        print("\nFATAL ERROR IN PROCESS_QUERY:")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

# 4. Expose the Server to the Internet
ngrok_token = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(ngrok_token)

# Close any existing tunnels to prevent errors
ngrok.kill()

public_url = ngrok.connect(8000).public_url
print(f"SECURENOC API IS LIVE AT: {public_url}/ask")

# 5. Start the Server in a Background Thread
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print("Background thread started. API is ready to receive requests!")

SECURENOC API IS LIVE AT: https://mia-propertied-cristopher.ngrok-free.dev/ask
Background thread started. API is ready to receive requests!


INFO:     Started server process [23216]


In [ ]:
import requests

try:
    print("Pinging local API...")
    res = requests.post("http://127.0.0.1:8000/ask", json={"query": "How fix BGP error"})
    print(f"Status Code: {res.status_code}")
    print(f"Response: {res.text}")
except Exception as e:
    print(f"Failed: {e}")

Pinging local API...

Received query from frontend: How fix BGP error
  [Diagnostics] Router applied filters: None

FATAL ERROR IN PROCESS_QUERY:
INFO:     127.0.0.1:43618 - "POST /ask HTTP/1.1" 500 Internal Server Error
Status Code: 500
Response: {"detail":"no such table: sops"}


Traceback (most recent call last):
  File "/tmp/ipykernel_23216/540375633.py", line 37, in ask_securenoc
    ans, srcs = process_query(request.query)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23216/675134997.py", line 77, in process_query
    cursor.execute(f"SELECT sop_id, full_json FROM sops WHERE sop_id IN ({placeholders})", top_ids)
sqlite3.OperationalError: no such table: sops
